# MT-GINO: Modular Reference Workflow

This notebook delegates implementation to `FLOW-reconstruction-in-VPM-using-FNO/FINAL`. The preserved baseline is a shared GNO/FNO latent with separate particle residual and Eulerian field decoders. State is `[x,y,z,Gamma_x,Gamma_y,Gamma_z,sigma]`; the base step is `x_next=x+dt*u` and `Gamma_next=Gamma+dt*(gradU @ Gamma)`, followed by the learned seven-channel residual. Optional mechanisms are listed as independent switches below.

In [ ]:
from pathlib import Path
import sys
import torch
ROOT = Path.cwd()
FINAL = ROOT / 'FLOW-reconstruction-in-VPM-using-FNO' / 'FINAL'
sys.path.insert(0, str(FINAL))
from gino.data.dataset import load_processed_dataset, EvolutionDataset
from gino.data.normalization import NormalizationStats
from gino.model.gino import GINOSharedLatent, build_latent_grid
from gino.dynamics.state_transition import predict_next_state
from gino.dynamics.rollout import autoregressive_rollout
from gino.training.losses import combined_loss, normalized_rollout_loss
from gino.training.scheduled_sampling import scheduled_sampling_probability
from gino.training.noise import random_walk_noise
from gino.training.pushforward import pushforward_sequence
from gino.training.trainer import Trainer
from gino.evaluation.one_step import evaluate_one_step
from gino.evaluation.autoregressive import evaluate_autoregressive


## Data and train-only normalization

In [ ]:
DATASET = (FINAL / 'configs/default.yaml')  # Set the processed NPZ path in the canonical config.
# data = load_processed_dataset(dataset_path)
# Build whole-sequence train/validation/test IDs and fit statistics on train IDs only.


## Model and baseline losses
The architecture and forward pass live in `gino.model`; this notebook does not duplicate them.

In [ ]:
# model = GINOSharedLatent(input_channels, 7, field_channels, global_channels, model_config)
# latent_grid = build_latent_grid(model_config['latent_res'], device)
# state/field loss uses the existing homoscedastic formulation in gino.training.losses.


## Independent optional mechanisms
- Short or longer rollout loss: configure `use_rollout_loss` and horizon curriculum.
- Scheduled sampling: configure `use_scheduled_sampling`; source behavior only replaces predicted velocity/gradient inputs.
- GNS random walk: configure `use_gns_noise`, `gns_initial_std`, `gns_walk_std`; residual targets must be rebuilt consistently.
- Pushforward: configure `use_pushforward`; subsequent inputs come from detached model predictions.
- Task adapters, attention, architectural skips, and global conditioning are separate model switches.


## Training, validation, checkpointing, evaluation

In [ ]:
# The same trainer and evaluators are used by scripts/train.py and scripts/evaluate.py.
# trainer = Trainer(model, latent_grid, train_ds, val_ds, stats, training_config, run_dir, device)
# history = trainer.fit(resolved_config, metadata)
# Validation selects checkpoints; test results are reported only after selection.
# For closed-loop rollout pass a rebuild callback that recomputes geometry from every predicted xyz.
